In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'gmlp_mcmc/inference'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# Soay Vital-Rate Inference With The 760-Location DeepRV Decoder

For every GP vital rate, NumPyro samples a 760-dimensional standard-normal latent vector. DeepRV produces the complete RBF GP draw at all 760 CSV `z` locations before the relevant observation rows are selected. The original LogNormal lengthscale prior is truncated to the decoder's training range `[0.05, 10]`. Recruitment remains intercept-only and sex probability remains fixed at 0.5.

In [ ]:
import hashlib
import json
import sys
from pathlib import Path
import time
import jax
import jax.numpy as jnp
import numpy as np
import numpyro
import numpyro.distributions as dist
import pandas as pd
from jax import random
from numpyro.infer import MCMC, NUTS

repo_root = PROJECT_ROOT
sys.path.insert(0, str(repo_root))

from dl4bi.vae.train_utils import generate_surrogate_decoder
from experiments.gmlp_mcmc.train_gmlp_deeprv_soay_760 import TrainConfig, load_checkpoint, load_locations

In [ ]:
data_path = GENERATED / 'soay/size_only/data/gp_ibm_dataset.csv'
inference_dir = GENERATED / 'gmlp_mcmc/inference'
decoder_dir = GENERATED / 'gmlp_mcmc/training/gmlp_deeprv_rbf_soay_760'
metadata_path = decoder_dir / "metadata_soay_760.json"
model_path = decoder_dir / "model_soay_760.ckpt"

#Recover decoder configuration
config = TrainConfig(data_path=str(data_path))
#Load trained decoder
model, state = load_checkpoint(model_path, config)

metadata = json.loads(metadata_path.read_text())
decoder = generate_surrogate_decoder(state, model)

print("Checkpoint:", model_path)
print("Training Step:", int(state.step))

In [ ]:
df = pd.read_csv(data_path)
s_full = jnp.asarray(df["z"].to_numpy())[:, None]
np.testing.assert_array_equal(np.asarray(s_full), np.asarray(load_locations(config)))

def observed_rows(column):
    observed = df[column].notna().to_numpy()
    return jnp.asarray(np.flatnonzero(observed)), jnp.asarray(df.loc[observed, column].to_numpy())

idx_surv, y_surv = observed_rows("Surv")
idx_repr, y_repr = observed_rows("Repr")
idx_growth, y_growth = observed_rows("z1")
idx_recr, y_recr = observed_rows("Recr")
idx_rcsz, y_rcsz = observed_rows("Rcsz")
sex_probability = 0.5
print({"Survival": len(idx_surv), "Reproduction": len(idx_repr), "Growth": len(idx_growth), "Recruitment": len(idx_recr), "Recruit Size": len(idx_rcsz), "Sex Probability": sex_probability})

In [ ]:
def lengthscale_prior():
    log_lengthscale = dist.TruncatedNormal(0, 1, low=jnp.log(config.lengthscale_min), high=jnp.log(config.lengthscale_max))
    return dist.TransformedDistribution(log_lengthscale, dist.transforms.ExpTransform())

# Decoder is trained to sample a unit variance, 0 mean GP draw
def full_deeprv_draw(s, decoder, variance, lengthscale):
    latent = numpyro.sample("latent", dist.Normal(0, 1).expand([len(s)]).to_event(1))
    unit_draw = decoder(latent[None, :], jnp.array([lengthscale]), s=s).squeeze((0, 2))
    return jnp.sqrt(variance) * unit_draw

def gp_binary_deeprv(s, idx, y, decoder):
    intercept = numpyro.sample("int", dist.Normal(0, 5))
    variance = numpyro.sample("var", dist.HalfNormal(2))
    lengthscale = numpyro.sample("ls", lengthscale_prior())
    mu_full = numpyro.deterministic("mu_full", full_deeprv_draw(s, decoder, variance, lengthscale) + intercept)
    mu = numpyro.deterministic("mu", mu_full[idx])
    numpyro.sample("y", dist.Bernoulli(logits=mu), obs=y)

def gp_gaussian_deeprv(s, idx, y, decoder):
    intercept = numpyro.sample("int", dist.Normal(0, 5))
    variance = numpyro.sample("var", dist.HalfNormal(2))
    lengthscale = numpyro.sample("ls", lengthscale_prior())
    noise = numpyro.sample("noise", dist.HalfNormal(1))
    mu_full = numpyro.deterministic("mu_full", full_deeprv_draw(s, decoder, variance, lengthscale) + intercept)
    mu = numpyro.deterministic("mu", mu_full[idx])
    numpyro.sample("y", dist.Normal(mu, noise), obs=y)

def intercept_binary(y):
    intercept = numpyro.sample("int", dist.Normal(0, 5))
    numpyro.sample("y", dist.Bernoulli(logits=intercept), obs=y)

def fit(model, key, **kwargs):
    mcmc = MCMC(NUTS(model), num_warmup=num_warmup, num_samples=num_samples, num_chains=num_chains)
    start = time.perf_counter()
    mcmc.run(key, **kwargs)
    samples = mcmc.get_samples()
    jax.tree.map(lambda x: x.block_until_ready(), samples)
    elapsed = time.perf_counter() - start
    mcmc.print_summary(exclude_deterministic=True)
    return samples, elapsed

In [ ]:
num_warmup = 1_000
num_samples = 2_500
num_chains = 2
keys = random.split(random.key(0), 5)
results_dir = inference_dir
results_dir.mkdir(parents=True, exist_ok=True)
timings_path = results_dir / "gpdf_gp_deeprv_soay_760_timings.json"
timings = {"num_warmup": num_warmup, "num_samples": num_samples, "num_chains": num_chains}

In [ ]:
surv_gp, timings["survival_seconds"] = fit(gp_binary_deeprv, keys[0], s=s_full, idx=idx_surv, y=y_surv, decoder=decoder); 
timings_path.write_text(json.dumps(timings, indent=2))

In [ ]:
repr_gp, timings["reproduction_seconds"] = fit(gp_binary_deeprv, keys[1], s=s_full, idx=idx_repr, y=y_repr, decoder=decoder); timings_path.write_text(json.dumps(timings, indent=2))

In [ ]:
growth_gp, timings["growth_seconds"] = fit(gp_gaussian_deeprv, keys[2], s=s_full, idx=idx_growth, y=y_growth, decoder=decoder); timings_path.write_text(json.dumps(timings, indent=2))

In [ ]:
recr_samples, timings["recruitment_seconds"] = fit(intercept_binary, keys[3], y=y_recr); timings_path.write_text(json.dumps(timings, indent=2))

In [ ]:
rcsz_gp, timings["recruit_size_seconds"] = fit(gp_gaussian_deeprv, keys[4], s=s_full, idx=idx_rcsz, y=y_rcsz, decoder=decoder); timings["total_seconds"] = sum(v for k, v in timings.items() if k.endswith("_seconds")); timings_path.write_text(json.dumps(timings, indent=2))

In [ ]:
gpdf_gp_deeprv_samples = {
    "surv_int": surv_gp["int"], "surv_var": surv_gp["var"], "surv_ls": surv_gp["ls"], "surv_mu": surv_gp["mu"], "surv_mu_full": surv_gp["mu_full"], "surv_latent": surv_gp["latent"],
    "repr_int": repr_gp["int"], "repr_var": repr_gp["var"], "repr_ls": repr_gp["ls"], "repr_mu": repr_gp["mu"], "repr_mu_full": repr_gp["mu_full"], "repr_latent": repr_gp["latent"],
    "growth_int": growth_gp["int"], "growth_var": growth_gp["var"], "growth_ls": growth_gp["ls"], "growth_noise": growth_gp["noise"], "growth_mu": growth_gp["mu"], "growth_mu_full": growth_gp["mu_full"], "growth_latent": growth_gp["latent"],
    "recr_int": recr_samples["int"], "sex_prob": jnp.array(sex_probability),
    "rcsz_int": rcsz_gp["int"], "rcsz_var": rcsz_gp["var"], "rcsz_ls": rcsz_gp["ls"], "rcsz_noise": rcsz_gp["noise"], "rcsz_mu": rcsz_gp["mu"], "rcsz_mu_full": rcsz_gp["mu_full"], "rcsz_latent": rcsz_gp["latent"]
}

results_dir = inference_dir
results_dir.mkdir(parents=True, exist_ok=True)
samples_path = results_dir / "gpdf_gp_deeprv_soay_760_samples.npz"
np.savez_compressed(samples_path, **{name: np.asarray(jax.device_get(value)) for name, value in gpdf_gp_deeprv_samples.items()}) 
samples_path
